In [1]:
from pathlib import Path
import duckdb
import pandas as pd

# Project directories
ROOT = Path.cwd().parent

RAW_DIR = ROOT / "data" / "raw" / "parquet"
PROCESSED_DIR = ROOT / "data" / "processed"

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Connect to DuckDB
con = duckdb.connect()

print("Project root:", ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

print("\nNumber of Parquet files:", len(list(RAW_DIR.glob("*.parquet"))))

Project root: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra
Raw data directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\raw\parquet
Processed directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed

Number of Parquet files: 26


In [2]:
parquet_files = sorted(RAW_DIR.glob("*.parquet"))

for file in parquet_files:
    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{file.as_posix()}')"
    ).df()

    print(f"\n{'=' * 60}")
    print("FILE:", file.name)
    print(schema[["column_name", "column_type"]].to_string(index=False))


FILE: 2001.parquet
   column_name column_type
         State     VARCHAR
      District     VARCHAR
        Market     VARCHAR
     Commodity     VARCHAR
       Variety     VARCHAR
         Grade     VARCHAR
  Arrival_Date     VARCHAR
     Min_Price      BIGINT
     Max_Price      DOUBLE
   Modal_Price      BIGINT
Commodity_Code      BIGINT

FILE: 2002.parquet
   column_name column_type
         State     VARCHAR
      District     VARCHAR
        Market     VARCHAR
     Commodity     VARCHAR
       Variety     VARCHAR
         Grade     VARCHAR
  Arrival_Date     VARCHAR
     Min_Price      DOUBLE
     Max_Price      DOUBLE
   Modal_Price      DOUBLE
Commodity_Code      BIGINT

FILE: 2003.parquet
   column_name column_type
         State     VARCHAR
      District     VARCHAR
        Market     VARCHAR
     Commodity     VARCHAR
       Variety     VARCHAR
         Grade     VARCHAR
  Arrival_Date     VARCHAR
     Min_Price      DOUBLE
     Max_Price      DOUBLE
   Modal_Price      DO

In [3]:
# Read all annual Parquet files together
file_paths = [str(file).replace("\\", "/") for file in parquet_files]

file_list = "[" + ", ".join(
    "'" + path + "'" for path in file_paths
) + "]"

con.execute(f"""
    CREATE OR REPLACE VIEW raw_data AS
    SELECT *
    FROM read_parquet({file_list}, union_by_name = true)
""")

# Inspect combined schema
schema = con.execute("""
    DESCRIBE SELECT * FROM raw_data
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,State,VARCHAR,YES,None,None,None
1,District,VARCHAR,YES,None,None,None
2,Market,VARCHAR,YES,None,None,None
3,Commodity,VARCHAR,YES,None,None,None
4,Variety,VARCHAR,YES,None,None,None
5,Grade,VARCHAR,YES,None,None,None
6,Arrival_Date,VARCHAR,YES,None,None,None
7,Min_Price,DOUBLE,YES,None,None,None
8,Max_Price,DOUBLE,YES,None,None,None
9,Modal_Price,DOUBLE,YES,None,None,None


In [4]:
# Check total records and date range
overview = con.execute("""
    SELECT
        COUNT(*) AS Total_Records,
        MIN(TRY_CAST(Arrival_Date AS DATE)) AS First_Date,
        MAX(TRY_CAST(Arrival_Date AS DATE)) AS Last_Date,
        COUNT(DISTINCT Commodity) AS Unique_Commodities,
        COUNT(DISTINCT State) AS Unique_States
    FROM raw_data
""").df()

display(overview)

,Total_Records,First_Date,Last_Date,Unique_Commodities,Unique_States
0,75984017,2001-01-10,2026-04-21,389,34


In [5]:
commodity_audit = con.execute("""
    SELECT
        COUNT(DISTINCT Commodity) AS Original_Unique_Names,

        COUNT(DISTINCT LOWER(TRIM(Commodity)))
            AS Normalized_Unique_Names,

        COUNT(*) FILTER (
            WHERE Commodity IS NULL OR TRIM(Commodity) = ''
        ) AS Missing_Commodity_Records

    FROM raw_data
""").df()

display(commodity_audit)

,Original_Unique_Names,Normalized_Unique_Names,Missing_Commodity_Records
0,389,387,0


In [6]:
commodity_variants = con.execute("""
    SELECT
        LOWER(TRIM(Commodity)) AS Normalized_Name,
        COUNT(DISTINCT Commodity) AS Original_Variants,
        STRING_AGG(DISTINCT Commodity, ' | ') AS Original_Names

    FROM raw_data
    WHERE Commodity IS NOT NULL
    GROUP BY LOWER(TRIM(Commodity))
    HAVING COUNT(DISTINCT Commodity) > 1
    ORDER BY Original_Variants DESC
""").df()

display(commodity_variants)

,Normalized_Name,Original_Variants,Original_Names
0,tulip,2,tulip | Tulip
1,kutki,2,Kutki | kutki


In [7]:
con.execute("""
    CREATE OR REPLACE VIEW selected_raw AS

    SELECT
        *,
        CASE
            WHEN LOWER(TRIM(Commodity)) = 'soyabean'
                THEN 'Soybean'
            ELSE TRIM(Commodity)
        END AS Project_Commodity

    FROM raw_data

    WHERE LOWER(TRIM(Commodity)) IN (
        'onion',
        'tomato',
        'soyabean',
        'cotton',
        'grapes',
        'pomegranate'
    )
""")

In [8]:
selected_summary = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT State) AS Unique_States,
        COUNT(DISTINCT Market) AS Unique_Markets,
        MIN(TRY_CAST(Arrival_Date AS DATE)) AS First_Date,
        MAX(TRY_CAST(Arrival_Date AS DATE)) AS Last_Date

    FROM selected_raw

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(selected_summary)

,Project_Commodity,Total_Records,Unique_States,Unique_Markets,First_Date,Last_Date
0,Cotton,936032,24,1437,2001-04-17,2026-04-20
1,Grapes,369784,30,971,2001-01-27,2026-04-20
2,Onion,3168490,32,3180,2001-01-10,2026-04-20
3,Pomegranate,572365,25,854,2001-11-17,2026-04-20
4,Soybean,1192838,25,1446,2001-03-12,2026-04-20
5,Tomato,2818577,32,3033,2001-02-08,2026-04-20


In [9]:
date_price_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE TRY_CAST(Arrival_Date AS DATE) IS NULL
        ) AS Invalid_Dates,

        COUNT(*) FILTER (
            WHERE Min_Price IS NULL
        ) AS Missing_Min_Price,

        COUNT(*) FILTER (
            WHERE Max_Price IS NULL
        ) AS Missing_Max_Price,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Missing_Modal_Price,

        COUNT(*) FILTER (
            WHERE Modal_Price < 0
        ) AS Negative_Modal_Price,

        COUNT(*) FILTER (
            WHERE Min_Price > Max_Price
        ) AS Invalid_Price_Ranges

    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(date_price_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Records,Invalid_Dates,Missing_Min_Price,Missing_Max_Price,Missing_Modal_Price,Negative_Modal_Price,Invalid_Price_Ranges
0,Cotton,936032,0,0,1,0,0,97
1,Grapes,369784,0,0,0,0,0,132
2,Onion,3168490,0,0,1,0,0,2133
3,Pomegranate,572365,0,0,0,0,0,133
4,Soybean,1192838,0,0,0,0,0,255
5,Tomato,2818577,0,0,0,0,0,2309


In [10]:
range_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) FILTER (
            WHERE Modal_Price < Min_Price
               OR Modal_Price > Max_Price
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
              AND Max_Price = 0
              AND Modal_Price > 0
        ) AS Zero_Range_With_Positive_Modal,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
               OR Max_Price = 0
               OR Modal_Price = 0
        ) AS Records_With_Any_Zero_Price

    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(range_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Modal_Outside_Range,Zero_Range_With_Positive_Modal,Records_With_Any_Zero_Price
0,Cotton,3707,3609,3790
1,Grapes,2347,2215,3006
2,Onion,125287,122479,129314
3,Pomegranate,6439,6306,7504
4,Soybean,3263,494,2775
5,Tomato,86320,84010,88060


In [11]:
remaining_issues = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Arrival_Date,
        Min_Price,
        Modal_Price,
        Max_Price

    FROM selected_raw

    WHERE (
        Modal_Price < Min_Price
        OR Modal_Price > Max_Price
    )
    AND NOT (
        Min_Price = 0
        AND Max_Price = 0
        AND Modal_Price > 0
    )

    ORDER BY Project_Commodity, Arrival_Date

    LIMIT 30
""").df()

display(remaining_issues)

,Project_Commodity,State,District,Market,Variety,Arrival_Date,Min_Price,Modal_Price,Max_Price
0,Cotton,Karnataka,Mysore,Nanjangud,Other,2007-12-07,1550.0,1550.0,0.0
1,Cotton,Karnataka,Mysore,Nanjangud,Other,2007-12-11,2125.0,2125.0,0.0
2,Cotton,Karnataka,Mysore,Nanjangud,Other,2007-12-12,1850.0,1850.0,0.0
3,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-01-07,1210.0,1210.0,0.0
4,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-08-19,3355.0,3355.0,0.0
5,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-08-27,3100.0,3100.0,0.0
6,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-08-30,1400.0,1400.0,0.0
7,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-09-02,1770.0,1770.0,0.0
8,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-09-04,2515.0,2515.0,0.0
9,Cotton,Karnataka,Mysore,Nanjangud,Other,2008-10-30,2600.0,2600.0,0.0


In [12]:
remaining_pattern_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) AS Total_Remaining_Issues,

        COUNT(*) FILTER (
            WHERE Max_Price = 0 AND Min_Price > 0
        ) AS Zero_Max_Positive_Min,

        COUNT(*) FILTER (
            WHERE Min_Price = 0 AND Max_Price > 0
        ) AS Zero_Min_Positive_Max,

        COUNT(*) FILTER (
            WHERE Min_Price > Max_Price
        ) AS Min_Greater_Than_Max,

        COUNT(*) FILTER (
            WHERE Min_Price > 0
              AND Max_Price > 0
              AND Modal_Price > 0
              AND (
                  Modal_Price < Min_Price
                  OR Modal_Price > Max_Price
              )
        ) AS Modal_Outside_Nonzero_Range

    FROM selected_raw

    WHERE (
        Modal_Price < Min_Price
        OR Modal_Price > Max_Price
    )
    AND NOT (
        Min_Price = 0
        AND Max_Price = 0
        AND Modal_Price > 0
    )

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(remaining_pattern_audit)

,Project_Commodity,Total_Remaining_Issues,Zero_Max_Positive_Min,Zero_Min_Positive_Max,Min_Greater_Than_Max,Modal_Outside_Nonzero_Range
0,Cotton,98,97,0,97,1
1,Grapes,132,132,0,132,0
2,Onion,2808,2129,0,2133,679
3,Pomegranate,133,133,0,133,0
4,Soybean,2769,255,0,255,2514
5,Tomato,2310,2306,0,2309,4


In [13]:
nonzero_range_samples = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Arrival_Date,
        Min_Price,
        Modal_Price,
        Max_Price

    FROM selected_raw

    WHERE Min_Price > 0
      AND Max_Price > 0
      AND Modal_Price > 0
      AND (
          Modal_Price < Min_Price
          OR Modal_Price > Max_Price
      )

    ORDER BY Project_Commodity, Arrival_Date

    LIMIT 50
""").df()

display(nonzero_range_samples)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Variety,Arrival_Date,Min_Price,Modal_Price,Max_Price
0,Cotton,Karnataka,Dharwad,Hubli (Amaragol) APMC,Jayadhar,2026-04-01,7350.0,1800.0,7800.0
1,Onion,Madhya Pradesh,Ratlam,Ratlam,Nasik,2023-12-21,610.0,1750.0,610.0
2,Onion,Madhya Pradesh,Mandsaur,Mandsaur,Local,2023-12-21,1433.0,200.0,1521.0
3,Onion,Madhya Pradesh,Shajapur,Sajapur,Onion,2023-12-21,1852.0,1027.0,1852.0
4,Onion,Madhya Pradesh,Ratlam,Sailana,Nasik,2023-12-21,1200.0,351.0,1745.0
5,Onion,Madhya Pradesh,Neemuch,Neemuch,Onion,2023-12-22,1000.0,713.0,1401.0
6,Onion,Madhya Pradesh,Indore,Indore,White,2023-12-22,1611.0,1193.0,1611.0
7,Onion,Madhya Pradesh,Ratlam,Sailana,Onion,2023-12-22,1400.0,600.0,1599.0
8,Onion,Madhya Pradesh,Indore,Indore,White,2023-12-23,1392.0,1086.0,1392.0
9,Onion,Madhya Pradesh,Neemuch,Neemuch,Red,2023-12-26,700.0,100.0,1400.0


In [14]:
severity_audit = con.execute("""
    SELECT
        Project_Commodity,

        COUNT(*) AS Anomaly_Count,

        ROUND(
            AVG(
                ABS(Modal_Price - Min_Price)
                / NULLIF(Min_Price, 0) * 100
            ), 2
        ) AS Avg_Difference_From_Min_Pct,

        ROUND(
            MAX(
                ABS(Modal_Price - Min_Price)
                / NULLIF(Min_Price, 0) * 100
            ), 2
        ) AS Max_Difference_From_Min_Pct

    FROM selected_raw

    WHERE Min_Price > 0
      AND Max_Price > 0
      AND Modal_Price > 0
      AND (
          Modal_Price < Min_Price
          OR Modal_Price > Max_Price
      )

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(severity_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Anomaly_Count,Avg_Difference_From_Min_Pct,Max_Difference_From_Min_Pct
0,Cotton,1,75.51,75.51
1,Onion,679,188.73,20816.67
2,Soybean,2514,21.99,10011.11
3,Tomato,4,10.57,37.50


In [15]:
duplicate_audit = con.execute("""
    SELECT
        COUNT(*) AS Rows_In_Duplicate_Groups,
        COUNT(DISTINCT duplicate_key) AS Duplicate_Groups
    FROM (
        SELECT
            HASH(
                Project_Commodity,
                State,
                District,
                Market,
                Variety,
                Grade,
                TRY_CAST(Arrival_Date AS DATE),
                Min_Price,
                Max_Price,
                Modal_Price,
                Commodity_Code
            ) AS duplicate_key

        FROM selected_raw

        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            TRY_CAST(Arrival_Date AS DATE),
            Min_Price,
            Max_Price,
            Modal_Price,
            Commodity_Code

        HAVING COUNT(*) > 1
    )
""").df()

display(duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Rows_In_Duplicate_Groups,Duplicate_Groups
0,219,219


In [16]:
duplicate_audit = con.execute("""
    WITH duplicate_groups AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date,
            Min_Price,
            Max_Price,
            Modal_Price,
            Commodity_Code,
            COUNT(*) AS Group_Size

        FROM selected_raw

        GROUP BY ALL

        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS Duplicate_Groups,

        SUM(Group_Size) AS Rows_In_Duplicate_Groups,

        SUM(Group_Size - 1) AS Excess_Duplicate_Rows,

        MAX(Group_Size) AS Largest_Group_Size

    FROM duplicate_groups
""").df()

display(duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Duplicate_Groups,Rows_In_Duplicate_Groups,Excess_Duplicate_Rows,Largest_Group_Size
0,219,438.0,219.0,2


In [17]:
exact_duplicate_audit = con.execute("""
    WITH duplicate_groups AS (
        SELECT
            State,
            District,
            Market,
            Commodity,
            Variety,
            Grade,
            Arrival_Date,
            Min_Price,
            Max_Price,
            Modal_Price,
            Commodity_Code,
            COUNT(*) AS Group_Size

        FROM selected_raw

        GROUP BY ALL

        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS Exact_Duplicate_Groups,
        SUM(Group_Size) AS Rows_In_Exact_Duplicate_Groups,
        SUM(Group_Size - 1) AS Excess_Exact_Duplicate_Rows,
        MAX(Group_Size) AS Largest_Group_Size

    FROM duplicate_groups
""").df()

display(exact_duplicate_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Exact_Duplicate_Groups,Rows_In_Exact_Duplicate_Groups,Excess_Exact_Duplicate_Rows,Largest_Group_Size
0,219,438.0,219.0,2


In [18]:
con.execute("""
    CREATE OR REPLACE VIEW processed_data AS

    SELECT DISTINCT
        TRIM(State) AS State,
        TRIM(District) AS District,
        TRIM(Market) AS Market,
        TRIM(Commodity) AS Source_Commodity,

        CASE
            WHEN LOWER(TRIM(Commodity)) = 'soyabean'
            THEN 'Soybean'
            ELSE TRIM(Commodity)
        END AS Project_Commodity,

        TRIM(Variety) AS Variety,
        TRIM(Grade) AS Grade,

        Arrival_Date,
        TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date,

        Min_Price,
        Max_Price,
        Modal_Price,
        Commodity_Code,

        -- Missing and invalid price flags
        Modal_Price IS NULL AS flag_missing_modal,

        Modal_Price < 0 AS flag_negative_modal,

        -- Zero and incomplete price ranges
        (
            Min_Price = 0
            AND Max_Price = 0
            AND Modal_Price > 0
        ) AS flag_zero_range_positive_modal,

        (
            Min_Price > 0
            AND Max_Price = 0
        ) AS flag_zero_max_positive_min,

        (
            Min_Price = 0
            AND Max_Price > 0
        ) AS flag_zero_min_positive_max,

        (
            Min_Price = 0
            OR Max_Price = 0
        ) AS flag_incomplete_range,

        (
            Min_Price = 0
            OR Max_Price = 0
            OR Modal_Price = 0
        ) AS flag_any_price_zero,

        -- Invalid price relationships
        (
            Min_Price > Max_Price
        ) AS flag_min_gt_max,

        (
            Min_Price > 0
            AND Max_Price > 0
            AND Modal_Price > 0
            AND (
                Modal_Price < Min_Price
                OR Modal_Price > Max_Price
            )
        ) AS flag_modal_outside_range,

        -- Basic eligibility for modal-price forecasting
        (
            TRY_CAST(Arrival_Date AS DATE) IS NOT NULL
            AND Modal_Price > 0
        ) AS is_forecast_eligible

    FROM selected_raw
""")

print("Processed view created successfully.")

Processed view created successfully.


In [19]:
validation = con.execute("""
    SELECT
        COUNT(*) AS Processed_Records,

        COUNT(DISTINCT Project_Commodity) AS Commodities,

        MIN(Parsed_Date) AS First_Date,

        MAX(Parsed_Date) AS Last_Date,

        COUNT(*) FILTER (
            WHERE flag_missing_modal
        ) AS Missing_Modal_Records,

        COUNT(*) FILTER (
            WHERE flag_modal_outside_range
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE is_forecast_eligible
        ) AS Forecast_Eligible_Records

    FROM processed_data
""").df()

display(validation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Processed_Records,Commodities,First_Date,Last_Date,Missing_Modal_Records,Modal_Outside_Range,Forecast_Eligible_Records
0,9057867,6,2001-01-10,2026-04-20,0,3198,9057867


In [20]:
missing_value_audit = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Null_Modal_Prices,

        COUNT(*) FILTER (
            WHERE isnan(Modal_Price)
        ) AS NaN_Modal_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
               OR isnan(Modal_Price)
        ) AS Total_Missing_Modal_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price > 0
              AND NOT isnan(Modal_Price)
              AND Modal_Price IS NOT NULL
        ) AS Valid_Positive_Modal_Prices

    FROM selected_raw
""").df()

display(missing_value_audit)

,Null_Modal_Prices,NaN_Modal_Prices,Total_Missing_Modal_Prices,Valid_Positive_Modal_Prices
0,0,0,0,9058086


In [21]:
modal_check = con.execute("""
    SELECT
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Null_Prices,

        COUNT(*) FILTER (
            WHERE NOT isfinite(Modal_Price)
        ) AS Non_Finite_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price <= 0
        ) AS Non_Positive_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price > 0
              AND isfinite(Modal_Price)
        ) AS Positive_Finite_Prices

    FROM selected_raw
""").df()

display(modal_check)

,Total_Records,Null_Prices,Non_Finite_Prices,Non_Positive_Prices,Positive_Finite_Prices
0,9058086,0,0,0,9058086


In [22]:
modal_check_by_commodity = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Null_Prices,

        COUNT(*) FILTER (
            WHERE NOT isfinite(Modal_Price)
        ) AS Non_Finite_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price <= 0
        ) AS Non_Positive_Prices,

        COUNT(*) FILTER (
            WHERE Modal_Price > 0
              AND isfinite(Modal_Price)
        ) AS Positive_Finite_Prices

    FROM selected_raw
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(modal_check_by_commodity)

,Project_Commodity,Total_Records,Null_Prices,Non_Finite_Prices,Non_Positive_Prices,Positive_Finite_Prices
0,Cotton,936032,0,0,0,936032
1,Grapes,369784,0,0,0,369784
2,Onion,3168490,0,0,0,3168490
3,Pomegranate,572365,0,0,0,572365
4,Soybean,1192838,0,0,0,1192838
5,Tomato,2818577,0,0,0,2818577


In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW processed_data AS

    SELECT
        * REPLACE (
            (
                Parsed_Date IS NOT NULL
                AND Modal_Price IS NOT NULL
                AND isfinite(Modal_Price)
                AND Modal_Price > 0
            ) AS is_forecast_eligible
        )

    FROM processed_data
""")

print("Forecast eligibility rule updated.")

Forecast eligibility rule updated.


In [25]:
con.execute("""
    CREATE OR REPLACE VIEW processed_data AS

    SELECT DISTINCT
        TRIM(State) AS State,
        TRIM(District) AS District,
        TRIM(Market) AS Market,
        TRIM(Commodity) AS Source_Commodity,

        CASE
            WHEN LOWER(TRIM(Commodity)) = 'soyabean'
            THEN 'Soybean'
            ELSE TRIM(Commodity)
        END AS Project_Commodity,

        TRIM(Variety) AS Variety,
        TRIM(Grade) AS Grade,

        Arrival_Date,
        TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date,

        Min_Price,
        Max_Price,
        Modal_Price,
        Commodity_Code,

        Modal_Price IS NULL AS flag_missing_modal,
        Modal_Price < 0 AS flag_negative_modal,

        (
            Min_Price = 0 AND Max_Price = 0
            AND Modal_Price > 0
        ) AS flag_zero_range_positive_modal,

        (
            Min_Price > 0 AND Max_Price = 0
        ) AS flag_zero_max_positive_min,

        (
            Min_Price = 0 AND Max_Price > 0
        ) AS flag_zero_min_positive_max,

        (
            Min_Price = 0 OR Max_Price = 0
        ) AS flag_incomplete_range,

        (
            Min_Price = 0 OR Max_Price = 0 OR Modal_Price = 0
        ) AS flag_any_price_zero,

        Min_Price > Max_Price AS flag_min_gt_max,

        (
            Min_Price > 0
            AND Max_Price > 0
            AND Modal_Price > 0
            AND (
                Modal_Price < Min_Price
                OR Modal_Price > Max_Price
            )
        ) AS flag_modal_outside_range,

        (
            TRY_CAST(Arrival_Date AS DATE) IS NOT NULL
            AND Modal_Price IS NOT NULL
            AND isfinite(Modal_Price)
            AND Modal_Price > 0
        ) AS is_forecast_eligible

    FROM selected_raw
""")

print("Processed view successfully rebuilt.")

Processed view successfully rebuilt.


In [26]:
OUTPUT_FILE = PROCESSED_DIR / "mandi_prices_cleaned.parquet"

con.execute(f"""
    COPY processed_data
    TO '{OUTPUT_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", OUTPUT_FILE)
print("File exists:", OUTPUT_FILE.exists())
print("File size (MB):", round(OUTPUT_FILE.stat().st_size / (1024**2), 2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed\mandi_prices_cleaned.parquet
File exists: True
File size (MB): 94.52


In [27]:
saved_check = con.execute(f"""
    SELECT
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT Project_Commodity) AS Commodities,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date,
        COUNT(*) FILTER (
            WHERE is_forecast_eligible
        ) AS Forecast_Eligible_Records
    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
""").df()

display(saved_check)

,Total_Records,Commodities,First_Date,Last_Date,Forecast_Eligible_Records
0,9057867,6,2001-01-10,2026-04-20,9057867


In [28]:
market_summary = con.execute(f"""
    SELECT
        Project_Commodity,
        COUNT(*) AS Records,
        COUNT(DISTINCT State) AS States,
        COUNT(DISTINCT Market) AS Market_Names,
        COUNT(DISTINCT
            State || '|' || District || '|' || Market
        ) AS Market_Identities,
        COUNT(DISTINCT Variety) AS Varieties,
        COUNT(DISTINCT Grade) AS Grades

    FROM read_parquet('{OUTPUT_FILE.as_posix()}')

    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(market_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Records,States,Market_Names,Market_Identities,Varieties,Grades
0,Cotton,936024,24,1437,1459,109,11
1,Grapes,369772,30,971,980,10,13
2,Onion,3168401,32,3180,3223,26,14
3,Pomegranate,572345,25,854,859,3,12
4,Soybean,1192827,25,1446,1457,7,11
5,Tomato,2818498,32,3033,3075,6,14


In [29]:
final_audit = con.execute(f"""
    SELECT
        COUNT(*) AS Total_Records,

        COUNT(DISTINCT Project_Commodity) AS Commodities,

        COUNT(*) FILTER (
            WHERE Parsed_Date IS NULL
        ) AS Invalid_Dates,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
               OR NOT isfinite(Modal_Price)
               OR Modal_Price <= 0
        ) AS Invalid_Modal_Prices,

        COUNT(*) FILTER (
            WHERE flag_min_gt_max
        ) AS Invalid_Price_Ranges,

        COUNT(*) FILTER (
            WHERE flag_modal_outside_range
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE flag_zero_range_positive_modal
        ) AS Zero_Range_Positive_Modal,

        COUNT(*) FILTER (
            WHERE is_forecast_eligible
        ) AS Forecast_Eligible_Records

    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
""").df()

display(final_audit)

,Total_Records,Commodities,Invalid_Dates,Invalid_Modal_Prices,Invalid_Price_Ranges,Modal_Outside_Range,Zero_Range_Positive_Modal,Forecast_Eligible_Records
0,9057867,6,0,0,5059,3198,219113,9057867
